# Solution: Память оптимизатора

Считаем только постоянные данные на один параметр: параметр, его градиент, fp32-состояния оптимизатора и опциональную fp32 master copy. Активации и буферы зависят от модели и шага, поэтому в эту арифметику не входят.

Словари делают допустимые значения явными. Число слотов состояния умножается на 4, потому что momentum, `m` и `v` в модели задачи всегда fp32. Проверяем все строковые аргументы до сложения: так неверная конфигурация не даёт правдоподобного, но ошибочного числа.

In [1]:
def bytes_per_param(param_dtype, grad_dtype, optimizer, master_weights):
    dtype_bytes = {'fp32': 4, 'bf16': 2, 'fp16': 2}
    optimizer_slots = {'sgd': 0, 'sgd_momentum': 1, 'adamw': 2}

    try:
        parameter_bytes = dtype_bytes[param_dtype]
        gradient_bytes = dtype_bytes[grad_dtype]
        state_slots = optimizer_slots[optimizer]
    except KeyError as error:
        raise ValueError(f'Unsupported configuration value: {error.args[0]}') from None

    master_copy_bytes = 4 if master_weights else 0
    return parameter_bytes + gradient_bytes + state_slots * 4 + master_copy_bytes


In [2]:
examples = [
    ('fp32', 'fp32', 'adamw', False),
    ('bf16', 'bf16', 'adamw', True),
    ('fp32', 'fp32', 'sgd_momentum', False),
]
for configuration in examples:
    print(configuration, '->', bytes_per_param(*configuration), 'bytes/parameter')


('fp32', 'fp32', 'adamw', False) -> 16 bytes/parameter
('bf16', 'bf16', 'adamw', True) -> 16 bytes/parameter
('fp32', 'fp32', 'sgd_momentum', False) -> 12 bytes/parameter


In [3]:
from torch_judge import check
check('optimizer_memory_math')



🧪 Testing: Optimizer Memory Math (Easy)
──────────────────────────────────────────────────
  ✅ [1/5] fp32 AdamW without master weights (0.1ms)
  ✅ [2/5] bf16 mixed AdamW with master weights (0.0ms)
  ✅ [3/5] fp32 SGD without momentum (0.0ms)
  ✅ [4/5] fp32 SGD with momentum (0.0ms)
  ✅ [5/5] invalid dtype raises ValueError (0.0ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (0.2ms total)
  Progress saved. Run status() to see your dashboard.

